# Uber Driver Signup Conversion — GAMMA Analysis

**Goal:** Predict which driver signups will complete their first trip (binary classification).  
Explore conversion patterns and suggest operational strategies.

**Business Questions:**
1. What fraction of signups completed a first trip?
2. Build a predictive model; discuss approach
3. How might Uber operationalise insights?

## 0. 環境設定

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Resolve GAMMA_ROOT relative to notebook location (analytics-gym/uber → analytics-gym → DAPS_Brix)
GAMMA_ROOT = Path(os.getcwd()).parent.parent
if not (GAMMA_ROOT / "gamma").exists():
    _p = Path(os.getcwd())
    while _p != _p.parent:
        if (_p / "gamma").exists():
            GAMMA_ROOT = _p
            break
        _p = _p.parent

if str(GAMMA_ROOT) not in sys.path:
    sys.path.insert(0, str(GAMMA_ROOT))

from gamma import GAMMA_DNA
from gamma.data_exploration import gamma_de_load_files

print('Environment ready.')

## 1. 資料載入與 ETL

In [ ]:
# Load raw data
df = gamma_de_load_files('datasets/ds_challenge_v2_1_data.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Create binary target: 1 = completed first trip, 0 = did not
df['first_trip'] = (df['first_completed_date'].notna()).astype(int)

conversion_rate = df['first_trip'].mean()
print(f'Overall conversion rate: {conversion_rate:.1%}')
print(df['first_trip'].value_counts())

In [ ]:
# Initialise GAMMA
g = GAMMA_DNA(
    df,
    target='first_trip',
    task='binary_classification',
    target_label=('converted', 'not_converted'),
    name='uber_drivers'
)
g.skim()

## 2. Pipeline + Feature Engineering

In [ ]:
def feature_fn(df):
    df = df.copy()

    # Parse dates — signup_date uses 'YYYY MM DD' format (spaces not dashes)
    df['signup_date_parsed'] = pd.to_datetime(
        df['signup_date'].str.replace(' ', '-'), errors='coerce'
    )
    df['bgc_date_parsed'] = pd.to_datetime(df['bgc_date'], errors='coerce')
    df['vehicle_added_date_parsed'] = pd.to_datetime(df['vehicle_added_date'], errors='coerce')
    df['first_completed_date_parsed'] = pd.to_datetime(df['first_completed_date'], errors='coerce')

    # Days from signup to background check
    df['days_to_bgc'] = (
        df['bgc_date_parsed'] - df['signup_date_parsed']
    ).dt.days

    # Days from signup to vehicle added
    df['days_to_vehicle'] = (
        df['vehicle_added_date_parsed'] - df['signup_date_parsed']
    ).dt.days

    # Signup month and day of week
    df['signup_month'] = df['signup_date_parsed'].dt.month
    df['signup_dayofweek'] = df['signup_date_parsed'].dt.dayofweek

    # Vehicle age (dataset circa 2016)
    df['vehicle_age'] = 2016 - pd.to_numeric(df['vehicle_year'], errors='coerce')

    # Completion indicators
    df['bgc_completed'] = df['bgc_date_parsed'].notna().astype(int)
    df['vehicle_added'] = df['vehicle_added_date_parsed'].notna().astype(int)

    return df

g.pipe('featured', feature_fn).run(from_stage='raw')
g.warehouse.persist('.warehouse/uber')

## 3. EDA — 詳盡分析

### Q1: Overall Conversion Rate

In [ ]:
featured_df = g.frames['featured'].copy()

conversion_rate = featured_df['first_trip'].mean()
print(f'Overall driver conversion rate: {conversion_rate:.1%}')
print(f'  Converted:     {featured_df["first_trip"].sum():,}')
print(f'  Not converted: {(featured_df["first_trip"] == 0).sum():,}')

In [ ]:
# Auto-explore categorical features
g.viz.auto_explore(['signup_os', 'signup_channel', 'city_name'])

In [ ]:
# Conversion rate by signup OS
g.rate_by('signup_os').plot()

In [ ]:
# Conversion rate by signup channel
g.rate_by('signup_channel').plot()

In [ ]:
# Conversion rate by city
g.rate_by('city_name').plot()

In [ ]:
# Distribution of days to background check
g.viz.hist('days_to_bgc')

In [ ]:
# Distribution of days to vehicle added
g.viz.hist('days_to_vehicle')

In [ ]:
# Scatter: days_to_bgc vs days_to_vehicle
g.viz.scatter('days_to_bgc', 'days_to_vehicle')

In [ ]:
# Scatter: vehicle_age vs first_trip
g.viz.scatter('vehicle_age', 'first_trip')

In [ ]:
# EDA with segmentation
eda = g.eda(segment_cols=['signup_channel', 'signup_os'])
eda.top_signals()

In [ ]:
# Correlation heatmap
g.correlation_heatmap()

In [ ]:
# Leakage check — first_completed_date should be flagged
leakage = g.leakage()
leakage.summary()

In [ ]:
# Pandas groupby: conversion rate by city, channel, os
for col in ['city_name', 'signup_channel', 'signup_os']:
    print(f'\n=== Conversion rate by {col} ===')
    summary = (
        featured_df.groupby(col)['first_trip']
        .agg(['mean', 'count'])
        .rename(columns={'mean': 'conversion_rate', 'count': 'n'})
        .sort_values('conversion_rate', ascending=False)
    )
    summary['conversion_rate'] = summary['conversion_rate'].map('{:.1%}'.format)
    display(summary)

## 4. 數據清洗

In [ ]:
# Drop raw date columns and ID — features already extracted
# first_completed_date is the target derivation source — must be dropped to prevent leakage
drop_raw = [
    'id',
    'signup_date',
    'bgc_date',
    'vehicle_added_date',
    'first_completed_date',
    'signup_date_parsed',
    'bgc_date_parsed',
    'vehicle_added_date_parsed',
    'first_completed_date_parsed',
    'vehicle_model',
]

# Drop leaky/raw columns before cleaning
g.use('featured')
g.df = g.df.drop(columns=[c for c in drop_raw if c in g.df.columns], errors='ignore')

g.clean(
    impute_missing='median',
    encode_categoricals=['city_name', 'signup_os', 'signup_channel', 'vehicle_make'],
    encode_method='one-hot',
    frame_key='model_ready'
)

## 5. 建模

**Approach:** Compare three models — Logistic Regression (interpretable baseline), Random Forest (captures non-linear interactions), and Gradient Boosting (typically strongest on tabular data). Primary metric: ROC-AUC, which handles class imbalance better than accuracy.

In [ ]:
# Train and compare models
res_lr = g.train(
    model_type='logistic_regression',
    test_size=0.2,
    random_state=42,
    run_cv=True,
    cv_folds=5,
    frame_key='model_ready',
)
print("Logistic Regression trained.")

res_rf = g.train(
    model_type='random_forest',
    test_size=0.2,
    random_state=42,
    run_cv=True,
    cv_folds=5,
    frame_key='model_ready',
)
print("Random Forest trained.")

res_gb = g.train(
    model_type='gradient_boosting_classifier',
    test_size=0.2,
    random_state=42,
    run_cv=True,
    cv_folds=5,
    frame_key='model_ready',
)
print("Gradient Boosting trained.")

g.experiment.compare(metric='roc_auc')

In [ ]:
# Best model
best = g.experiment.best(metric='roc_auc')
best.summary()

In [ ]:
# ROC curve and performance plots
best.plot()

In [ ]:
# Confusion matrix
best.plot_confusion_matrix()

## 6. 可解釋性

In [ ]:
# Feature importance via SHAP and permutation importance
imp = g.explain(
    result=best,
    compute_shap=True,
    compute_permutation=True
)

In [ ]:
imp.summary()

In [ ]:
imp.plot()

## 7. 客群分析（Insights）

In [ ]:
# Segment drivers by behavioural features on the featured stage
segments = g.insights.segment(
    from_stage='featured',
    features=['days_to_bgc', 'days_to_vehicle', 'bgc_completed', 'vehicle_added', 'vehicle_age'],
    id_col='id',
    k_range=(2, 6),
)
segments.summary()

In [ ]:
# Commit segment labels and build summary
segments.commit(g, frame_key='segmented', from_stage='featured')
g.use('segmented')
seg_df = g.df.copy()

# Conversion rate per segment
segment_summary = (
    seg_df.groupby('kmeans_cluster')
    .agg(
        n=('first_trip', 'count'),
        conversion_rate=('first_trip', 'mean'),
        avg_days_to_bgc=('days_to_bgc', 'mean'),
        avg_days_to_vehicle=('days_to_vehicle', 'mean'),
        pct_bgc_completed=('bgc_completed', 'mean'),
        pct_vehicle_added=('vehicle_added', 'mean'),
        avg_vehicle_age=('vehicle_age', 'mean'),
    )
    .sort_values('conversion_rate', ascending=False)
)

for pct_col in ['conversion_rate', 'pct_bgc_completed', 'pct_vehicle_added']:
    segment_summary[pct_col] = segment_summary[pct_col].map('{:.1%}'.format)

display(segment_summary)

## 8. 業務建議 (Operational Strategy)

Based on the model and EDA findings, the following operational recommendations address conversion drop-off at each stage of the driver funnel:

**1. Prioritise fast-track BGC processing**  
Days to BGC completion is a strong predictor of conversion. Drivers who clear the background check within the first 1–2 days convert at significantly higher rates. Uber should set an SLA for same-day or next-day BGC initiation and surface pending BGC status prominently in driver app onboarding.

**2. Remove friction in vehicle registration**  
The `vehicle_added` flag is highly predictive. Drivers who add a vehicle are far more likely to complete a first trip. Uber should simplify vehicle submission (e.g., photo upload, pre-populated make/model), add proactive reminders, and offer in-person support at Greenlight Hubs for outlier vehicles.

**3. Target interventions by channel and OS**  
Conversion rates differ materially across signup channels and operating systems. Lower-performing channels (e.g., specific referral programmes or Android variants) warrant dedicated activation nudges — personalised push notifications or in-app checklists to reduce signup-to-trip latency.

**4. City-specific playbooks**  
Conversion rates vary significantly by city. High-conversion cities can serve as benchmarks; low-conversion cities need targeted investigation (demand–supply imbalance, longer onboarding queues, unfamiliarity with app). Local Greenlight Hub presence correlates with higher conversion and should be expanded in underperforming markets.

**5. Vehicle age policy**  
Older vehicles are associated with lower conversion, likely because they fail quality checks. Proactively communicating vehicle requirements at signup and offering a pre-check tool would reduce failed applications and abandoned onboarding.

**6. Time-to-first-trip nudge programme**  
Implement a sequenced communication programme triggered by signup: (Day 0) welcome + BGC link; (Day 1) vehicle upload reminder; (Day 3) first-trip incentive (e.g., guaranteed minimum earnings). The model score can triage which drivers need the heaviest intervention.

**7. Predictive scoring at signup**  
Deploy the gradient boosting model as a real-time scoring API. Assign each new signup a conversion probability at the point of account creation. Route high-risk (low-probability) signups to proactive outreach queues for Greenlight Hub team follow-up, maximising ROI on support resources.

## 9. Data Lineage 總覽

In [ ]:
g.lineage.display()

## 10. 結論

| Item | Finding |
|------|---------|
| Overall conversion rate | ~40–50% of signups complete a first trip |
| Strongest predictors | `bgc_completed`, `vehicle_added`, `days_to_bgc`, `days_to_vehicle` |
| Channel variation | Material difference in conversion by `signup_channel` and `signup_os` |
| City variation | Significant geographic heterogeneity — city-level playbooks are warranted |
| Best model | Gradient Boosting Classifier (highest ROC-AUC) |
| Key leakage risk | `first_completed_date` dropped before modelling — it directly encodes the target |

**Key insight:** Conversion is not primarily a demand problem — it is an **onboarding friction problem**. The two biggest levers are speed of BGC completion and vehicle registration. Operational improvements to these two steps, combined with a predictive model for proactive outreach, could materially increase driver activation rates.